In [1]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.optim as optim


In [2]:
# ==========================================
# 1. ДАННЫЕ (Финтех домен)
# ==========================================
train_texts = [
    "потерял карту, заблокируйте срочно",
    "украли деньги со счета, помогите",
    "как заблокировать кредитку?",
    "какая ставка по ипотеке на новостройку?",
    "хочу взять кредит наличными",
    "как оформить автокредит?",
    "где ближайший банкомат?",
    "до скольки работает отделение на пушкина?",
    "где снять наличные без комиссии?"
]
train_labels = [0, 0, 0, 1, 1, 1, 2, 2, 2]



In [3]:
# ==========================================
# 2. ТОКЕНИЗАЦИЯ И ПОДГОТОВКА
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = 'cointegrated/rubert-tiny2'

# TODO 1: Загрузи токенизатор для указанной модели с помощью метода AutoTokenizer.from_pretrained
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# TODO 2: Токенизируй список train_texts с помощью загруженного токенизатора.
# Подсказка: передай внутрь списка текстов аргументы padding=True, truncation=True, max_length=32, return_tensors='pt'.
# В результате ты получишь словарь (dict), внутри которого будут тензоры 'input_ids' и 'attention_mask'.
encoded_inputs = tokenizer(
    train_texts,
    padding=True,
    return_tensors="pt"
).to(device)

# Извлекаем тензоры из словаря токенизатора
input_ids = encoded_inputs['input_ids']
attention_mask = encoded_inputs['attention_mask']
labels = torch.tensor(train_labels).to(device)

# Создаем DataLoader для батчирования
dataset = TensorDataset(input_ids, attention_mask, labels)
dataloader = DataLoader(dataset, batch_size=3, shuffle=True)


config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.74M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [4]:
# ==========================================
# 3. ИНИЦИАЛИЗАЦИЯ BERT И ОПТИМИЗАТОРА
# ==========================================
# TODO 3: Загрузи предобученную модель с помощью AutoModelForSequenceClassification.from_pretrained.
# Не забудь передать имя модели и аргумент num_labels=3 (количество наших классов).
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)

# TODO 4: Создай оптимизатор AdamW (torch.optim.AdamW).
# Передай ему параметры модели (model.parameters()) и learning rate = 2e-5 (это стандарт для Fine-tuning).
optimizer = optim.AdamW(model.parameters(), lr=2e-5)


model.safetensors: reconstructing file:   0%|          |  0.00B /  118MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
# ==========================================
# 4. ЦИКЛ ДООБУЧЕНИЯ (FINE-TUNING)
# ==========================================
EPOCHS = 5
model.train() # Переводим модель в режим обучения

print("Начинаем Fine-tuning...")
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in dataloader:
        b_input_ids, b_attention_mask, b_labels = batch

        # TODO 5: Сделай forward pass.
        # Фишка HuggingFace: если передать в модель аргументы input_ids, attention_mask и labels,
        # она САМА вычислит CrossEntropyLoss!
        # Вызови модель, передав эти три тензора. Результат сохрани в переменную outputs.
        outputs = model(input_ids=b_input_ids,
                        attention_mask=b_attention_mask,
                        labels=b_labels
                        )

        # TODO 6: Извлеки значение ошибки из outputs (обратись к атрибуту .loss).
        loss = outputs.loss

        # TODO 7: Очисти градиенты оптимизатора (.zero_grad()).
        optimizer.zero_grad()

        # TODO 8: Выполни обратное распространение ошибки (.backward()) и шаг оптимизатора (.step()).
        loss.backward()
        total_loss += loss.item()
        optimizer.step()

    avg_loss = total_loss / len(dataloader)
    print(f"Эпоха {epoch+1}/{EPOCHS}, Loss: {avg_loss:.4f}")




Начинаем Fine-tuning...
Эпоха 1/5, Loss: 1.1074
Эпоха 2/5, Loss: 1.0875
Эпоха 3/5, Loss: 1.0761
Эпоха 4/5, Loss: 1.0687
Эпоха 5/5, Loss: 1.0602


In [7]:
# ==========================================
# 5. ИНФЕРЕНС (Классификация нового текста)
# ==========================================
test_text = "подскажите адрес вашего главного офиса"
print(f"\nТестируем на новой фразе: '{test_text}'")

model.eval() # Переводим модель в режим предсказания
with torch.no_grad():
    # TODO 9: Токенизируй test_text точно так же, как в TODO 2, сохранив в test_encoded.
    test_encoded = tokenizer(
        [test_text],
        padding=True,
        return_tensors="pt"
    ).to(device)

    # TODO 10: Сделай forward pass. Передай в модель test_encoded['input_ids'] и test_encoded['attention_mask'].
    # ВАЖНО: labels передавать НЕ нужно, ведь мы их не знаем!
    test_outputs = model(
        input_ids=test_encoded['input_ids'],
        attention_mask=test_encoded['attention_mask']
    )

    # TODO 11: Извлеки сырые оценки (логиты) из test_outputs (обратись к атрибуту .logits).
    logits = test_outputs.logits

    # TODO 12: Найди номер предсказанного класса с помощью torch.argmax(logits, dim=1).
    pred_class = torch.argmax(logits, dim=1)

    classes_names = {0: "Безопасность", 1: "Кредиты", 2: "Отделения"}
    print(f"Предсказанный класс: {classes_names[pred_class.item()]}")


Тестируем на новой фразе: 'подскажите адрес вашего главного офиса'
Предсказанный класс: Отделения
